# CiteScope — SciBERT mejorado (experimento 07)

Este notebook conserva intacto `05_scibert.ipynb` y prueba mejoras controladas sobre su resultado de referencia (**Macro F1 de validación = 0,6836**).

Las mejoras son: entrada estructurada con presupuesto de tokens por campo, búsqueda pequeña de *learning rate*, evaluación con varias semillas y un *ensemble* opcional con Logistic Regression. El conjunto de **test no se utiliza** en este notebook.

## 1. Dependencias y configuración

Este experimento requiere `torch`, `transformers`, `accelerate`, `pandas`, `numpy` y `scikit-learn`. El entrenamiento puede tardar varios minutos u horas según el dispositivo.

In [1]:
import gc
import os
import random
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score

try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

MODEL_NAME = "allenai/scibert_scivocab_uncased"
REFERENCE_F1 = 0.6836
TARGET = "citing_primary_category"
MAX_LEN = 512

# Búsqueda moderada: tres entrenamientos. Para una prueba rápida usa [2e-5].
LEARNING_RATES = [1e-5, 2e-5, 3e-5]
RUN_SEED_CHECK = True
SEARCH_SEED = 42
MAX_EPOCHS = 5

# Después de elegir el learning rate, comprueba estabilidad con estas semillas.
RUN_SEED_CHECK = True
STABILITY_SEEDS = [17, 42, 73]

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"torch={torch.__version__} | dispositivo={device}")

torch=2.13.0+cu132 | dispositivo=cuda


## 2. Carga reproducible de datos

Se localiza la raíz del repositorio sin depender de que Jupyter haya sido abierto exactamente desde `models/`. Se reutiliza la asignación anti-fuga de `split_assignment.csv`.

In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "models").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del repositorio.")


REPO_ROOT = find_repo_root(Path.cwd())
DATASET_DIR = REPO_ROOT / "Dataset"
ARTIFACTS_DIR = REPO_ROOT / "models" / "artifacts"
CANONICAL = DATASET_DIR / "unarxive_microproyecto.jsonl"
LOCAL_COPY = DATASET_DIR / "copy_unarxive_microproyecto.jsonl"
DATA_PATH = CANONICAL if CANONICAL.exists() else LOCAL_COPY

if not DATA_PATH.exists():
    raise FileNotFoundError("No se encontró el dataset. Ejecuta dvc pull.")

df = pd.read_json(DATA_PATH, lines=True, dtype={"citing_arxiv_id": "string"})
split_map = pd.read_csv(ARTIFACTS_DIR / "split_assignment.csv")[["citation_id", "split"]]
df = df.merge(split_map, on="citation_id", how="left", validate="one_to_one")
assert df["split"].notna().all(), "Hay registros sin split."
assert set(df["split"]) == {"train", "val", "test"}

labels = sorted(df[TARGET].unique())
label2id = {label: index for index, label in enumerate(labels)}
id2label = {index: label for label, index in label2id.items()}
df["label"] = df[TARGET].map(label2id)

train = df[df["split"] == "train"].reset_index(drop=True)
val = df[df["split"] == "val"].reset_index(drop=True)
print(f"Dataset: {DATA_PATH.name}")
print(f"train={len(train)} | val={len(val)} | test reservado={(df['split'] == 'test').sum()}")

Dataset: unarxive_microproyecto.jsonl
train=2400 | val=800 | test reservado=800


## 3. Entrada estructurada y presupuesto de tokens

El notebook 05 concatena todos los campos y trunca el resultado completo. Aquí se reserva espacio para cada fuente:

- Contexto: 192 tokens.
- Título: 48 tokens.
- Abstract: 268 tokens.
- Tokens especiales: 4.

Así, título y abstract no desaparecen cuando el contexto es largo.

In [3]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
CTX_BUDGET = 192
TITLE_BUDGET = 48
ABSTRACT_BUDGET = 268
assert 1 + CTX_BUDGET + 1 + TITLE_BUDGET + 1 + ABSTRACT_BUDGET + 1 == MAX_LEN


def field_tokens(value, budget):
    text = "" if pd.isna(value) else str(value).strip()
    return tokenizer.encode(text, add_special_tokens=False, truncation=True, max_length=budget)


def encode_row(row):
    context_ids = field_tokens(row["citation_context"], CTX_BUDGET)
    title_ids = field_tokens(row["cited_title"], TITLE_BUDGET)
    abstract_ids = field_tokens(row["cited_abstract"], ABSTRACT_BUDGET)

    input_ids = (
        [tokenizer.cls_token_id]
        + context_ids + [tokenizer.sep_token_id]
        + title_ids + [tokenizer.sep_token_id]
        + abstract_ids + [tokenizer.sep_token_id]
    )
    # Segmento 0: contexto. Segmento 1: metadatos del artículo citado.
    first_segment = 1 + len(context_ids) + 1
    token_type_ids = [0] * first_segment + [1] * (len(input_ids) - first_segment)
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "token_type_ids": token_type_ids,
        "labels": int(row["label"]),
    }


class StructuredDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.items = [encode_row(row) for _, row in frame.iterrows()]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


train_ds = StructuredDataset(train)
val_ds = StructuredDataset(val)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
lengths = np.array([len(item["input_ids"]) for item in train_ds.items])
print(f"Longitud media={lengths.mean():.1f} | p95={np.percentile(lengths, 95):.0f} | máximo={lengths.max()}")

d:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre\Grado_Microproyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Longitud media=326.7 | p95=466 | máximo=489


## 4. Funciones de entrenamiento

Cada corrida parte del mismo modelo preentrenado, registra la mejor época y utiliza Macro F1 para seleccionar el checkpoint.

In [4]:
from transformers import (
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = logits.argmax(axis=-1)
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "accuracy": accuracy_score(y_true, y_pred),
    }


def train_once(learning_rate, seed, run_name):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
    )
    output_dir = ARTIFACTS_DIR / f"scibert07_{run_name}_ckpt"
    args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=MAX_EPOCHS,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.10,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        seed=seed,
        data_seed=seed,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    train_output = trainer.train()
    prediction = trainer.predict(val_ds)
    logits = prediction.predictions
    y_pred = logits.argmax(axis=-1)
    y_true = val["label"].to_numpy()
    row = {
        "run_name": run_name,
        "learning_rate": learning_rate,
        "seed": seed,
        "macro_f1_val": f1_score(y_true, y_pred, average="macro"),
        "accuracy_val": accuracy_score(y_true, y_pred),
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "epochs_completed": train_output.metrics.get("epoch"),
        "train_runtime_seconds": train_output.metrics.get("train_runtime"),
    }
    return trainer, logits, row


def release_trainer(trainer):
    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

## 5. Búsqueda pequeña de learning rate

Se mantienen constantes la entrada y la semilla. Solo cambia el *learning rate*. Esto permite atribuir la diferencia a ese parámetro.

In [5]:
lr_rows = []
for lr in LEARNING_RATES:
    run_name = f"lr_{lr:.0e}_seed_{SEARCH_SEED}"
    print(f"\n=== {run_name} ===")
    trainer, _, row = train_once(lr, SEARCH_SEED, run_name)
    lr_rows.append(row)
    print(f"Macro F1 val={row['macro_f1_val']:.4f}")
    release_trainer(trainer)

lr_results = pd.DataFrame(lr_rows).sort_values("macro_f1_val", ascending=False).reset_index(drop=True)
best_lr = float(lr_results.loc[0, "learning_rate"])
print("\nResultados:")
print(lr_results.to_string(index=False))
print(f"\nMejor learning rate: {best_lr:g}")


=== lr_1e-05_seed_42 ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.120700,1.065412,0.642690,0.643750
2,0.804100,0.960797,0.685319,0.682500
3,0.614500,0.958732,0.695831,0.695000
4,0.457800,1.023875,0.679654,0.676250
5,0.380800,1.045063,0.677080,0.675000


Macro F1 val=0.6958

=== lr_2e-05_seed_42 ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.031200,0.961255,0.666566,0.671250
2,0.707800,0.961522,0.673250,0.672500
3,0.440600,1.092307,0.677068,0.677500
4,0.253200,1.326379,0.665980,0.662500
5,0.129300,1.375598,0.670563,0.668750


Macro F1 val=0.6771

=== lr_3e-05_seed_42 ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.024100,0.974419,0.665168,0.665000
2,0.653000,1.046566,0.651167,0.652500
3,0.408700,1.202791,0.668012,0.666250
4,0.216100,1.547614,0.645838,0.645000
5,0.068300,1.652761,0.647440,0.645000


Macro F1 val=0.6680

Resultados:
        run_name  learning_rate  seed  macro_f1_val  accuracy_val                                                                                                                                             best_checkpoint  best_metric  epochs_completed  train_runtime_seconds
lr_1e-05_seed_42        0.00001    42      0.695831       0.69500 D:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre\Grado_Microproyecto\models\artifacts\scibert07_lr_1e-05_seed_42_ckpt\checkpoint-900     0.695831               5.0               522.9308
lr_2e-05_seed_42        0.00002    42      0.677068       0.67750 D:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre\Grado_Microproyecto\models\artifacts\scibert07_lr_2e-05_seed_42_ckpt\checkpoint-900     0.677068               5.0               539.0456
lr_3e-05_seed_42        0.00003    42      0.668012       0.66625 D:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre\G

## 6. Estabilidad entre semillas

Una mejora aislada puede depender del azar. Se repite la mejor configuración con tres semillas y se reportan media y desviación estándar. Para ahorrar tiempo, establece `RUN_SEED_CHECK = False` al inicio.

In [6]:
seed_rows = []
best_trainer = None
best_logits = None
best_row = None

seeds_to_run = STABILITY_SEEDS if RUN_SEED_CHECK else [SEARCH_SEED]
for seed in seeds_to_run:
    run_name = f"best_lr_{best_lr:.0e}_seed_{seed}"
    print(f"\n=== {run_name} ===")
    trainer, logits, row = train_once(best_lr, seed, run_name)
    seed_rows.append(row)
    if best_row is None or row["macro_f1_val"] > best_row["macro_f1_val"]:
        if best_trainer is not None:
            release_trainer(best_trainer)
        best_trainer, best_logits, best_row = trainer, logits, row
    else:
        release_trainer(trainer)

seed_results = pd.DataFrame(seed_rows).sort_values("macro_f1_val", ascending=False).reset_index(drop=True)
mean_f1 = seed_results["macro_f1_val"].mean()
std_f1 = seed_results["macro_f1_val"].std(ddof=1) if len(seed_results) > 1 else 0.0
print(seed_results.to_string(index=False))
print(f"\nMacro F1 medio={mean_f1:.4f} ± {std_f1:.4f}")
print(f"Mejor corrida={best_row['macro_f1_val']:.4f} | referencia 05={REFERENCE_F1:.4f}")


=== best_lr_1e-05_seed_17 ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.096700,1.070664,0.619460,0.633750
2,0.879200,0.939399,0.667863,0.670000
3,0.559500,0.966368,0.682982,0.686250
4,0.473300,1.023513,0.669212,0.670000
5,0.392700,1.042190,0.668962,0.668750



=== best_lr_1e-05_seed_42 ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.120700,1.065412,0.642690,0.643750
2,0.804100,0.960797,0.685319,0.682500
3,0.614500,0.958732,0.695831,0.695000
4,0.457800,1.023875,0.679654,0.676250
5,0.380800,1.045063,0.677080,0.675000



=== best_lr_1e-05_seed_73 ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.111100,1.047312,0.649204,0.651250
2,0.775100,0.923126,0.682160,0.682500
3,0.526700,0.963586,0.675718,0.673750
4,0.499600,0.998653,0.665717,0.666250


             run_name  learning_rate  seed  macro_f1_val  accuracy_val                                                                                                                                                  best_checkpoint  best_metric  epochs_completed  train_runtime_seconds
best_lr_1e-05_seed_42        0.00001    42      0.695831       0.69500 D:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre\Grado_Microproyecto\models\artifacts\scibert07_best_lr_1e-05_seed_42_ckpt\checkpoint-900     0.695831               5.0               525.8651
best_lr_1e-05_seed_17        0.00001    17      0.682982       0.68625 D:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre\Grado_Microproyecto\models\artifacts\scibert07_best_lr_1e-05_seed_17_ckpt\checkpoint-900     0.682982               5.0               521.9703
best_lr_1e-05_seed_73        0.00001    73      0.682160       0.68250 D:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre

## 7. Ensemble SciBERT + Logistic Regression

Se combinan las probabilidades de SciBERT con las de un modelo TF-IDF + Logistic Regression. El peso se selecciona solamente en validación. `alpha=1` representa SciBERT solo.

In [7]:
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def enriched_text(frame):
    return [
        "\n".join(
            part for part in (
                str(row.citation_context or "").strip(),
                "" if pd.isna(row.cited_title) else str(row.cited_title).strip(),
                "" if pd.isna(row.cited_abstract) else str(row.cited_abstract).strip(),
            ) if part
        )
        for row in frame.itertuples(index=False)
    ]


classic = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=SEARCH_SEED)),
])
classic.fit(enriched_text(train), train[TARGET])
classic_probs = classic.predict_proba(enriched_text(val))

# Reordenar las columnas clásicas para que coincidan con label2id.
classic_order = [list(classic.classes_).index(label) for label in labels]
classic_probs = classic_probs[:, classic_order]
scibert_probs = softmax(best_logits, axis=1)
y_true = val["label"].to_numpy()

ensemble_rows = []
for alpha in np.arange(0.0, 1.01, 0.1):
    blended = alpha * scibert_probs + (1 - alpha) * classic_probs
    pred = blended.argmax(axis=1)
    ensemble_rows.append({
        "alpha_scibert": round(float(alpha), 1),
        "macro_f1_val": f1_score(y_true, pred, average="macro"),
        "accuracy_val": accuracy_score(y_true, pred),
    })

ensemble_results = pd.DataFrame(ensemble_rows).sort_values("macro_f1_val", ascending=False).reset_index(drop=True)
best_alpha = float(ensemble_results.loc[0, "alpha_scibert"])
print(ensemble_results.to_string(index=False))
print(f"\nMejor alpha SciBERT={best_alpha:.1f} | Macro F1={ensemble_results.loc[0, 'macro_f1_val']:.4f}")

 alpha_scibert  macro_f1_val  accuracy_val
           0.9      0.698072       0.69750
           0.8      0.698072       0.69750
           0.6      0.697945       0.69750
           0.5      0.697325       0.69750
           0.7      0.696828       0.69625
           1.0      0.695831       0.69500
           0.4      0.693792       0.69375
           0.3      0.691095       0.69125
           0.2      0.683910       0.68375
           0.1      0.671150       0.67250
           0.0      0.627744       0.62875

Mejor alpha SciBERT=0.9 | Macro F1=0.6981


## 8. Reporte del mejor resultado

Se presenta el desempeño por clase del mejor ensemble. Esto sigue siendo una medición de validación, no el resultado final de test.

In [8]:
best_blended = best_alpha * scibert_probs + (1 - best_alpha) * classic_probs
best_pred = best_blended.argmax(axis=1)
print(classification_report(y_true, best_pred, target_names=labels, digits=3))
print(f"Referencia notebook 05: {REFERENCE_F1:.4f}")
print(f"Mejor SciBERT 07:      {best_row['macro_f1_val']:.4f}")
print(f"Mejor ensemble 07:     {f1_score(y_true, best_pred, average='macro'):.4f}")

              precision    recall  f1-score   support

       cs.AI      0.580     0.580     0.580       100
       cs.CL      0.729     0.780     0.754       100
       cs.CV      0.688     0.860     0.764       100
       cs.IR      0.723     0.680     0.701       100
       cs.LG      0.471     0.490     0.480       100
       cs.MA      0.774     0.720     0.746       100
       cs.NE      0.814     0.700     0.753       100
       cs.RO      0.846     0.770     0.806       100

    accuracy                          0.698       800
   macro avg      0.703     0.698     0.698       800
weighted avg      0.703     0.698     0.698       800

Referencia notebook 05: 0.6836
Mejor SciBERT 07:      0.6958
Mejor ensemble 07:     0.6981


## 9. Persistencia de resultados

Se guardan resultados separados de los del notebook 05. Los checkpoints permanecen ignorados por Git y posteriormente deben registrarse mediante DVC o MLflow.

In [9]:
ARTIFACTS_DIR.mkdir(exist_ok=True)
lr_results.to_csv(ARTIFACTS_DIR / "scibert07_lr_search.csv", index=False)
seed_results.to_csv(ARTIFACTS_DIR / "scibert07_seed_results.csv", index=False)
ensemble_results.to_csv(ARTIFACTS_DIR / "scibert07_ensemble_results.csv", index=False)

summary = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "input_strategy": "structured_token_budgets",
    "context_budget": CTX_BUDGET,
    "title_budget": TITLE_BUDGET,
    "abstract_budget": ABSTRACT_BUDGET,
    "best_learning_rate": best_lr,
    "mean_macro_f1_seeds": mean_f1,
    "std_macro_f1_seeds": std_f1,
    "best_scibert_macro_f1_val": best_row["macro_f1_val"],
    "best_ensemble_alpha": best_alpha,
    "best_ensemble_macro_f1_val": f1_score(y_true, best_pred, average="macro"),
    "reference_05_macro_f1_val": REFERENCE_F1,
    "eval_split": "val",
}])
summary.to_csv(ARTIFACTS_DIR / "scibert07_summary.csv", index=False)
print(summary.to_string(index=False))
print("\nTest continúa reservado. No fue evaluado en este notebook.")

                      model_name           input_strategy  context_budget  title_budget  abstract_budget  best_learning_rate  mean_macro_f1_seeds  std_macro_f1_seeds  best_scibert_macro_f1_val  best_ensemble_alpha  best_ensemble_macro_f1_val  reference_05_macro_f1_val eval_split
allenai/scibert_scivocab_uncased structured_token_budgets             192            48              268             0.00001             0.686991            0.007667                   0.695831                  0.9                    0.698072                     0.6836        val

Test continúa reservado. No fue evaluado en este notebook.


## 10. Registro de resultados en MLflow

Registro retrospectivo de las corridas ya ejecutadas.

In [1]:
import mlflow
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "http://100.58.157.126:5000"
MLFLOW_EXPERIMENT_NAME = "CiteScope - SciBERT Plus"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

client = MlflowClient()

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

d:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre\Grado_Microproyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/08/29 18:58:00 INFO mlflow.tracking.fluent: Experiment with name 'CiteScope - SciBERT Plus' does not exist. Creating a new experiment.


Tracking URI: http://100.58.157.126:5000
Experimento: CiteScope - SciBERT Plus
Experiment ID: 2


### 10.1 Registro de búsqueda del *learning rate*

Se registran en MLflow las corridas utilizadas para comparar diferentes valores de *learning rate*. Cada ejecución almacena su configuración y sus métricas de validación, permitiendo identificar qué velocidad de aprendizaje produjo el mejor resultado en SciBERT Plus. El registro evita crear corridas duplicadas.

In [2]:
from pathlib import Path

import pandas as pd

# Localizar la raíz aunque el notebook se abra desde otra carpeta.
def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "models" / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del repositorio.")


repo_root_mlflow = find_repo_root(Path.cwd())
artifacts_dir_mlflow = repo_root_mlflow / "models" / "artifacts"
lr_csv = artifacts_dir_mlflow / "scibert07_lr_search.csv"

lr_mlflow = pd.read_csv(lr_csv)

created = 0
skipped = 0

for _, row in lr_mlflow.iterrows():
    import_key = f"scibert_plus_lr_{row['learning_rate']:.0e}_seed_{int(row['seed'])}"

    existing = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.import_key = '{import_key}'",
    )

    if not existing.empty:
        print("Omitida porque ya existe:", import_key)
        skipped += 1
        continue

    with mlflow.start_run(run_name=str(row["run_name"])):
        mlflow.set_tags({
            "project": "CiteScope",
            "model_family": "SciBERT",
            "phase": "learning_rate_search",
            "source_notebook": "07_scibert_plus.ipynb",
            "eval_split": "validation",
            "import_key": import_key,
        })

        mlflow.log_params({
            "model_name": "allenai/scibert_scivocab_uncased",
            "input_strategy": "structured_token_budgets",
            "learning_rate": float(row["learning_rate"]),
            "seed": int(row["seed"]),
            "max_epochs": 5,
            "max_length": 512,
            "context_budget": 192,
            "title_budget": 48,
            "abstract_budget": 268,
        })

        mlflow.log_metrics({
            "macro_f1_val": float(row["macro_f1_val"]),
            "accuracy_val": float(row["accuracy_val"]),
            "best_metric": float(row["best_metric"]),
            "epochs_completed": float(row["epochs_completed"]),
            "train_runtime_seconds": float(row["train_runtime_seconds"]),
        })

        print("Registrada:", row["run_name"])
        created += 1

print(f"\nCreadas: {created} | Omitidas: {skipped}")

Registrada: lr_1e-05_seed_42
🏃 View run lr_1e-05_seed_42 at: http://100.58.157.126:5000/#/experiments/2/runs/e78501c94da9482ea3f265eaf110b3f0
🧪 View experiment at: http://100.58.157.126:5000/#/experiments/2
Registrada: lr_2e-05_seed_42
🏃 View run lr_2e-05_seed_42 at: http://100.58.157.126:5000/#/experiments/2/runs/3f9217631a30424880da8caacac61314
🧪 View experiment at: http://100.58.157.126:5000/#/experiments/2
Registrada: lr_3e-05_seed_42
🏃 View run lr_3e-05_seed_42 at: http://100.58.157.126:5000/#/experiments/2/runs/441a644b3bed45cc874382aada2f4971
🧪 View experiment at: http://100.58.157.126:5000/#/experiments/2

Creadas: 3 | Omitidas: 0


### 10.2 Registro de estabilidad entre semillas

Se registran en MLflow las corridas realizadas con el mejor *learning rate* y diferentes semillas. Esto permite comparar la estabilidad de SciBERT Plus y determinar cuánto varía su rendimiento entre entrenamientos. Cada corrida conserva sus parámetros, métricas y etiquetas, evitando registros duplicados.

In [3]:
seed_csv = artifacts_dir_mlflow / "scibert07_seed_results.csv"
seed_mlflow = pd.read_csv(seed_csv)

created = 0
skipped = 0

for _, row in seed_mlflow.iterrows():
    import_key = f"scibert_plus_stability_lr_{row['learning_rate']:.0e}_seed_{int(row['seed'])}"

    existing = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.import_key = '{import_key}'",
    )

    if not existing.empty:
        print("Omitida porque ya existe:", import_key)
        skipped += 1
        continue

    with mlflow.start_run(run_name=str(row["run_name"])):
        mlflow.set_tags({
            "project": "CiteScope",
            "model_family": "SciBERT",
            "phase": "seed_stability",
            "source_notebook": "07_scibert_plus.ipynb",
            "eval_split": "validation",
            "import_key": import_key,
        })

        mlflow.log_params({
            "model_name": "allenai/scibert_scivocab_uncased",
            "input_strategy": "structured_token_budgets",
            "learning_rate": float(row["learning_rate"]),
            "seed": int(row["seed"]),
            "max_epochs": 5,
            "max_length": 512,
            "context_budget": 192,
            "title_budget": 48,
            "abstract_budget": 268,
        })

        mlflow.log_metrics({
            "macro_f1_val": float(row["macro_f1_val"]),
            "accuracy_val": float(row["accuracy_val"]),
            "best_metric": float(row["best_metric"]),
            "epochs_completed": float(row["epochs_completed"]),
            "train_runtime_seconds": float(row["train_runtime_seconds"]),
        })

        print("Registrada:", row["run_name"])
        created += 1

print(f"\nCreadas: {created} | Omitidas: {skipped}")

Registrada: best_lr_1e-05_seed_42
🏃 View run best_lr_1e-05_seed_42 at: http://100.58.157.126:5000/#/experiments/2/runs/db729b611fe04dd68785d934384b2b83
🧪 View experiment at: http://100.58.157.126:5000/#/experiments/2
Registrada: best_lr_1e-05_seed_17
🏃 View run best_lr_1e-05_seed_17 at: http://100.58.157.126:5000/#/experiments/2/runs/b69bc75d27344221988f66d8abf4293d
🧪 View experiment at: http://100.58.157.126:5000/#/experiments/2
Registrada: best_lr_1e-05_seed_73
🏃 View run best_lr_1e-05_seed_73 at: http://100.58.157.126:5000/#/experiments/2/runs/02e8c89971c54e0eb5ca410331812672
🧪 View experiment at: http://100.58.157.126:5000/#/experiments/2

Creadas: 3 | Omitidas: 0


### 10.3 Registro del resumen final y artefactos

Se crea una corrida de resumen que consolida la mejor configuración de SciBERT Plus, su estabilidad entre semillas y el resultado del ensemble. También se almacenan en MLflow los archivos CSV generados durante el experimento para conservar la trazabilidad de los resultados.

In [4]:
summary_csv = artifacts_dir_mlflow / "scibert07_summary.csv"
ensemble_csv = artifacts_dir_mlflow / "scibert07_ensemble_results.csv"

summary_mlflow = pd.read_csv(summary_csv)
ensemble_mlflow = pd.read_csv(ensemble_csv)

summary_row = summary_mlflow.iloc[0]
ensemble_best = ensemble_mlflow.sort_values(
    "macro_f1_val",
    ascending=False,
).iloc[0]

import_key = "scibert_plus_final_summary_v1"

existing = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.import_key = '{import_key}'",
)

if not existing.empty:
    print("El resumen ya está registrado:", import_key)
else:
    with mlflow.start_run(run_name="scibert_plus_final_summary") as run:
        mlflow.set_tags({
            "project": "CiteScope",
            "model_family": "SciBERT",
            "phase": "final_summary",
            "source_notebook": "07_scibert_plus.ipynb",
            "eval_split": "validation",
            "import_key": import_key,
        })

        mlflow.log_params({
            "model_name": summary_row["model_name"],
            "input_strategy": summary_row["input_strategy"],
            "best_learning_rate": float(summary_row["best_learning_rate"]),
            "best_ensemble_alpha": float(summary_row["best_ensemble_alpha"]),
            "context_budget": int(summary_row["context_budget"]),
            "title_budget": int(summary_row["title_budget"]),
            "abstract_budget": int(summary_row["abstract_budget"]),
            "max_length": 512,
            "number_of_classes": 8,
        })

        best_ensemble_f1 = float(
            summary_row["best_ensemble_macro_f1_val"]
        )
        reference_f1 = float(
            summary_row["reference_05_macro_f1_val"]
        )

        mlflow.log_metrics({
            "best_scibert_macro_f1_val": float(
                summary_row["best_scibert_macro_f1_val"]
            ),
            "best_ensemble_macro_f1_val": best_ensemble_f1,
            "best_ensemble_accuracy_val": float(
                ensemble_best["accuracy_val"]
            ),
            "mean_macro_f1_seeds": float(
                summary_row["mean_macro_f1_seeds"]
            ),
            "std_macro_f1_seeds": float(
                summary_row["std_macro_f1_seeds"]
            ),
            "reference_05_macro_f1_val": reference_f1,
            "improvement_over_reference": (
                best_ensemble_f1 - reference_f1
            ),
        })

        result_files = [
            artifacts_dir_mlflow / "scibert07_lr_search.csv",
            artifacts_dir_mlflow / "scibert07_seed_results.csv",
            artifacts_dir_mlflow / "scibert07_ensemble_results.csv",
            artifacts_dir_mlflow / "scibert07_summary.csv",
        ]

        for result_file in result_files:
            mlflow.log_artifact(
                str(result_file),
                artifact_path="results",
            )

        mlflow.log_dict(
            {
                "model_name": summary_row["model_name"],
                "labels": [
                    "cs.AI", "cs.CL", "cs.CV", "cs.IR",
                    "cs.LG", "cs.MA", "cs.NE", "cs.RO",
                ],
                "token_budgets": {
                    "context": int(summary_row["context_budget"]),
                    "title": int(summary_row["title_budget"]),
                    "abstract": int(summary_row["abstract_budget"]),
                    "max_length": 512,
                },
                "evaluation_split": "validation",
                "test_evaluated": False,
            },
            "configuration/model_config.json",
        )

        print("Resumen registrado")
        print("Run ID:", run.info.run_id)
        print("Artifact URI:", mlflow.get_artifact_uri())

Resumen registrado
Run ID: f30039ad604b4dd1a8442cc82cc2c879
Artifact URI: mlflow-artifacts:/2/f30039ad604b4dd1a8442cc82cc2c879/artifacts
🏃 View run scibert_plus_final_summary at: http://100.58.157.126:5000/#/experiments/2/runs/f30039ad604b4dd1a8442cc82cc2c879
🧪 View experiment at: http://100.58.157.126:5000/#/experiments/2


### 10.4 Registro del modelo seleccionado

Se registra en MLflow el checkpoint de SciBERT Plus con mejor Macro F1 de validación. El modelo incluye sus pesos, configuración y tokenizer, y queda versionado en S3 para que posteriormente pueda ser consumido por la API.

In [8]:
import gc

import mlflow.transformers
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

checkpoint_dir = (
    artifacts_dir_mlflow
    / "scibert07_best_lr_1e-05_seed_42_ckpt"
    / "checkpoint-900"
)

if not checkpoint_dir.is_dir():
    raise FileNotFoundError(
        f"No se encontró el checkpoint: {checkpoint_dir}"
    )

import_key = "scibert_plus_registered_model_v1"

existing = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.import_key = '{import_key}'",
)

successful_existing = existing[
    existing["status"] == "FINISHED"
]

if not successful_existing.empty:
    print("El modelo ya fue registrado:", import_key)
else:
    selected_model = (
        AutoModelForSequenceClassification.from_pretrained(
            checkpoint_dir
        )
    )
    selected_tokenizer = AutoTokenizer.from_pretrained(
        checkpoint_dir
    )

    with mlflow.start_run(
        run_name="scibert_plus_registered_model"
    ) as run:
        mlflow.set_tags({
            "project": "CiteScope",
            "model_family": "SciBERT",
            "phase": "model_registration",
            "source_notebook": "07_scibert_plus.ipynb",
            "eval_split": "validation",
            "import_key": import_key,
        })

        mlflow.log_params({
            "model_name": "allenai/scibert_scivocab_uncased",
            "checkpoint": "checkpoint-900",
            "learning_rate": 1e-5,
            "seed": 42,
            "max_length": 512,
            "context_budget": 192,
            "title_budget": 48,
            "abstract_budget": 268,
            "number_of_classes": 8,
        })

        mlflow.log_metrics({
            "macro_f1_val": 0.6958313167767745,
            "accuracy_val": 0.695,
        })

        model_info = mlflow.transformers.log_model(
            transformers_model={
                "model": selected_model,
                "tokenizer": selected_tokenizer,
            },
            name="model",
            task="text-classification",
            registered_model_name="CiteScope-SciBERT-Plus",
            metadata={
                "input_strategy": "structured_token_budgets",
                "context_budget": 192,
                "title_budget": 48,
                "abstract_budget": 268,
                "eval_split": "validation",
                "test_evaluated": False,
            },
            pip_requirements=[
                "torch==2.13.0",
                "transformers==4.57.1",
                "accelerate==1.14.0",
            ],
        )

        print("Modelo registrado")
        print("Run ID:", run.info.run_id)
        print("Model URI:", model_info.model_uri)

    del selected_model
    del selected_tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Device set to use cuda:0
2026/08/29 19:29:51 WARNING mlflow.transformers: The model card could not be retrieved from the hub due to Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'D:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre\Grado_Microproyecto\models\artifacts\scibert07_best_lr_1e-05_seed_42_ckpt\checkpoint-900'.
2026/08/29 19:29:51 WARNING mlflow.transformers: Unable to find license information for this model. Please verify permissible usage for the model you are storing prior to use.
Successfully registered model 'CiteScope-SciBERT-Plus'.
2026/08/29 19:39:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: CiteScope-SciBERT-Plus, version 1
Created version '1' of model 'CiteScope-SciBERT-Plus'.


Modelo registrado
Run ID: 21250d1982d14958a93c64321b8b42b9
Model URI: models:/m-313abcbb7b1140bcb57d5583a603a722
🏃 View run scibert_plus_registered_model at: http://100.58.157.126:5000/#/experiments/2/runs/21250d1982d14958a93c64321b8b42b9
🧪 View experiment at: http://100.58.157.126:5000/#/experiments/2


### 10.5 Marcado del modelo candidato

La versión registrada se identifica como candidata porque presenta el mejor resultado de validación, pero aún no ha sido evaluada sobre el conjunto de prueba. El alias permite que otros componentes, como la API, consulten el modelo sin depender de un número de versión fijo.

In [9]:
REGISTERED_MODEL_NAME = "CiteScope-SciBERT-Plus"
CANDIDATE_VERSION = "1"

client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="candidate",
    version=CANDIDATE_VERSION,
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=CANDIDATE_VERSION,
    key="validation_macro_f1",
    value="0.695831",
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=CANDIDATE_VERSION,
    key="test_evaluated",
    value="false",
)

candidate = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME,
    "candidate",
)

print("Modelo:", candidate.name)
print("Versión:", candidate.version)
print("Alias: candidate")
print(
    "URI para consumo:",
    f"models:/{REGISTERED_MODEL_NAME}@candidate",
)

Modelo: CiteScope-SciBERT-Plus
Versión: 1
Alias: candidate
URI para consumo: models:/CiteScope-SciBERT-Plus@candidate


### 10.6 Verificación de carga del modelo registrado

Se descarga la versión identificada con el alias `candidate` y se realiza una inferencia de prueba. Esta comprobación valida que el modelo almacenado en S3 puede recuperarse correctamente mediante MLflow. No utiliza el conjunto de prueba ni genera una métrica de evaluación.

In [10]:
candidate_uri = (
    "models:/CiteScope-SciBERT-Plus@candidate"
)

candidate_pipeline = mlflow.transformers.load_model(
    candidate_uri
)

smoke_prediction = candidate_pipeline(
    "This study applies a neural language model "
    "to classify scientific citation contexts.",
    truncation=True,
    max_length=512,
)

print("Modelo cargado desde:", candidate_uri)
print("Predicción técnica:", smoke_prediction)

2026/08/29 20:41:24 INFO mlflow.transformers: 'models:/CiteScope-SciBERT-Plus@candidate' resolved as 'mlflow-artifacts:/2/models/m-313abcbb7b1140bcb57d5583a603a722/artifacts'
`torch_dtype` is deprecated! Use `dtype` instead!
2026/08/29 20:41:26 WARNING mlflow.transformers.model_io: Could not specify device parameter for this pipeline type.Falling back to loading the model with the default device.
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


Modelo cargado desde: models:/CiteScope-SciBERT-Plus@candidate
Predicción técnica: [{'label': 'cs.IR', 'score': 0.7370373606681824}]
